# Brain demonstration

Dataset source: 

In [1]:
from pathlib import Path

from torchvision.models import ResNet18_Weights, resnet18
from torchvision.models.detection import (
    SSDLite320_MobileNet_V3_Large_Weights,
    ssdlite320_mobilenet_v3_large,
)

from hyppopipe.data.dataset import YAMLDataset
from hyppopipe.data.image import Image
from hyppopipe.pipeline import Pipeline, Step
from hyppopipe.pipeline.image.classification import ImageClassifier
from hyppopipe.pipeline.image.localization import ImageLocalizer
from hyppopipe.train import ModelCandidate, Trainer, TrainingConfig

In [3]:
brain_dataset = YAMLDataset(
    "../datasets/BrainTumor/dataset.yaml",
    strict=False,
)
split_data = brain_dataset.as_split_data()

In [4]:
brain_pipe = Pipeline(
    {
        "localize": Step(
            ImageLocalizer(),
            description="Detects ROI",
        ),
        "disease_classification": Step(
            ImageClassifier(source_mode="roi"),
            description="Classify Glioma, Meningioma, No Tumor or Pituitary",
        ),
    },
)

common_config = TrainingConfig(
    epochs=10,
    batch_size=16,
    val_batch_size=32,
    lr=3e-4,
    device="mps",
)

In [5]:
train_result = brain_pipe.train(
    {
        "localize": Trainer(
            model_candidates=[
                ModelCandidate(
                    ssdlite320_mobilenet_v3_large,
                    weights=[
                        SSDLite320_MobileNet_V3_Large_Weights.COCO_V1,
                    ],
                ),
            ],
        ),
        "disease_classification": Trainer(
            model_candidates=[
                ModelCandidate(resnet18, weights=[ResNet18_Weights.IMAGENET1K_V1]),
            ],
        ),
    },
    data=split_data,
    config=common_config,
)

19:19:16 INFO hyppopipe.pipeline.pipeline: Pipeline training started for steps: localize, disease_classification
19:19:16 INFO hyppopipe.train.trainer: Step 'localize': starting (ImageLocalizer); train=4737 val=512 epochs=10 batch_size=16 val_batch_size=32 device=mps
19:19:17 INFO hyppopipe.train.trainer: Step 'localize' model 'ssdlite320_mobilenet_v3_large_COCO_V1': training on mps (10 epochs, early_stopping=on, monitor=val_loss)
localize · ssdlite320_mobilenet_v3_large_COCO_V1:   0%|          | 0/10 [01:49<?, ?epoch/s, t=109.7s, train=3.8821, val=3.2335]19:21:07 INFO hyppopipe.train.trainer: Step 'localize' ssdlite320_mobilenet_v3_large_COCO_V1: epoch 1/10 train_loss=3.882120 val_loss=3.233495 best=3.233495 (epoch 109.68s)
localize · ssdlite320_mobilenet_v3_large_COCO_V1:  10%|█         | 1/10 [03:17<16:28, 109.84s/epoch, t=87.4s, train=1.7766, val=2.6870] 19:22:34 INFO hyppopipe.train.trainer: Step 'localize' ssdlite320_mobilenet_v3_large_COCO_V1: epoch 2/10 train_loss=1.776581 val_

In [6]:
train_result.export_artifacts(Path("artifacts/brain_tumor"), brain_pipe)

In [9]:
some_image = Image.from_path("../datasets/IM000014.jpg")
pred = brain_pipe.predict(
    some_image,
    bundle_path=Path("artifacts/brain_tumor"),
    device="mps",
)

In [ ]:
locy = pred.outputs["localize"]
# locy
locy.show(class_names=brain_dataset.classes, top_k=10)


# detections = {"boxes": boxes, "scores": scores, "labels": labels}
# for key, value in locy.detections.items():
#     print(value.shape)

#print()
# print(pred.outputs["disease_classification"])

# some_image = Image.from_path("datasets/IM000014.jpg")
# some_image.show()

# pred = brain_pipe.predict(some_image, bundle_path=Path("artifacts/brain"), device="mps")
# pred = brain_pipe.predict(some_image, bundle_path=Path("artifacts/brain"), device="mps")
# pred.outputs["localize"].crop.show()


/Users/nemo/diploma/Georgii-Lanin-gmlanin/src/hyppopipe/inference/types.py:169: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(**kwargs)
